<a href="https://colab.research.google.com/github/rohisenn/prompt-engn/blob/main/Medical_Expense_and_Health_Insurance_Claim_Analyzer_using_LLM_APIs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Medical Expense and Health Insurance Claim Analyzer using LLM APIs

This notebook demonstrates how to use the Gemini API to extract healthcare-related financial transactions from a given passage and classify them into appropriate categories, returning the output strictly in valid JSON format.

In [1]:
# Install the Google Generative AI library
!pip install -q google-generativeai

### Set up Gemini API
To use the Gemini API, you'll need an API key. If you don't already have one, create a key in Google AI Studio. In Colab, add the key to the secrets manager under the "🔑" in the left panel. Give it the name `GOOGLE_API_KEY`. Then pass the key to the SDK:

In [2]:
import google.generativeai as genai
from IPython.display import display, Markdown
import json
# Used to securely store your API key
from google.colab import userdata

GOOGLE_API_KEY=userdata.get('GOOGLE_API_KEY')
genai.configure(api_key=GOOGLE_API_KEY)

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


### Initialize the Generative Model
We will use `gemini-1.5-flash-latest` as requested by the user.

In [3]:
# Initialize the Gemini API with the specified model
gemini_model = genai.GenerativeModel('gemini-3.5-flash')

print("Gemini model initialized successfully!")

Gemini model initialized successfully!


### Define Input Data and Categories
Here's the passage containing medical expenses and the predefined categories for classification.

In [4]:
passage = """Ravi had several medical appointments during September. On 2 September, he visited Apollo Hospital for a doctor's consultation and paid ₹850 using his credit card. Later that day, he purchased prescribed medicines worth ₹1,420 from Apollo Pharmacy using UPI.

On 4 September, he underwent a blood test at Thyrocare costing ₹2,300. Two days later, he purchased a digital thermometer from MedPlus for ₹780 using his debit card.

On 8 September, Ravi attended a physiotherapy session costing ₹1,500. During the same week, he paid ₹6,800 as partial hospital charges after a minor procedure.

On 12 September, he renewed his health insurance policy by paying ₹12,500 through net banking. While returning home, he booked an Ola cab for ₹340.

Later that month, Ravi bought protein supplements costing ₹2,100 from HealthKart, spent ₹480 on lunch at Saravana Bhavan after a hospital visit, and finally purchased a blood pressure monitor worth ₹2,950."""

categories = [
    "Consultation",
    "Medicines",
    "Laboratory Tests",
    "Hospital Charges",
    "Insurance",
    "Medical Equipment",
    "Wellness",
    "Transportation",
    "Food",
    "Others"
]

# Define the expected JSON output format for clarity to the LLM
output_format = [
  {
    "date": "",
    "merchant": "",
    "amount": 0,
    "payment_method": "",
    "category": ""
  }
]

sample_output_record = {
  "date": "2 September",
  "merchant": "Apollo Hospital",
  "amount": 850,
  "payment_method": "Credit Card",
  "category": "Consultation"
}


### Construct the Prompt
The prompt is carefully designed to guide the LLM to extract the desired information, adhere to the specified categories, and return the output in a strict JSON format.

In [9]:
prompt = f"""You are an AI assistant specialized in extracting and classifying financial transactions related to healthcare. Your task is to analyze the provided passage, identify all healthcare-related financial transactions, and classify each into one of the specified categories. The output must be in a strict JSON array format.

Instructions:
- Extract all financial transactions related to healthcare from the passage.
- Classify each transaction into one of the following categories: {', '.join(categories)}.
- If the payment method is not explicitly mentioned, return "Unknown".
- The 'amount' field must contain only numeric values (integers).
- The output must be a JSON array of objects, with each object strictly following this structure: {json.dumps(output_format[0], indent=2)}
- Do not include any additional text or explanation outside of the JSON output.

Here is a sample output record:
{json.dumps(sample_output_record, indent=2)}

Passage:
{passage}

Return only the JSON array of transaction objects."""

### Generate Content using Gemini API
Now, let's call the Gemini model with the crafted prompt and display the extracted transactions.

In [10]:
# Generate content using the Gemini model
response = gemini_model.generate_content(prompt)

# Attempt to parse the response text as JSON
try:
    json_output = json.loads(response.text)
    display(Markdown("### Extracted Transactions (JSON Output):"))
    display(json_output)
except json.JSONDecodeError as e:
    display(Markdown("### Error: Could not parse JSON output from LLM."))
    display(Markdown(f"**Raw LLM Output:**\n```json\n{response.text}\n```"))
    display(f"Error details: {e}")

### Extracted Transactions (JSON Output):

[{'date': '2 September',
  'merchant': 'Apollo Hospital',
  'amount': 850,
  'payment_method': 'Credit Card',
  'category': 'Consultation'},
 {'date': '2 September',
  'merchant': 'Apollo Pharmacy',
  'amount': 1420,
  'payment_method': 'UPI',
  'category': 'Medicines'},
 {'date': '4 September',
  'merchant': 'Thyrocare',
  'amount': 2300,
  'payment_method': 'Unknown',
  'category': 'Laboratory Tests'},
 {'date': '6 September',
  'merchant': 'MedPlus',
  'amount': 780,
  'payment_method': 'Debit Card',
  'category': 'Medical Equipment'},
 {'date': '8 September',
  'merchant': 'Unknown',
  'amount': 1500,
  'payment_method': 'Unknown',
  'category': 'Wellness'},
 {'date': 'September',
  'merchant': 'Unknown',
  'amount': 6800,
  'payment_method': 'Unknown',
  'category': 'Hospital Charges'},
 {'date': '12 September',
  'merchant': 'Unknown',
  'amount': 12500,
  'payment_method': 'Net Banking',
  'category': 'Insurance'},
 {'date': '12 September',
  'merchant': 'Ola',
  'amount': 340,
